# V87 - Template-ID popularity

Pre-registered plan: v75 and v76 treated templates as noise to escape, but v76 proved templated chats beat assessment, so template identity may be signal. Predict each user by the mean relevance of same-template training users, falling back to global popularity for unseen groups. Same grouped folds under a fresh seed. Two verdicts: against global popularity (does template identity carry anything) and against word-only (does it lift the bar).

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 1313
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v87")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Template means vs popularity vs word-only

Group means use training folds only. Unseen or singleton groups fall back to the fold popularity.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

oof_t = np.zeros_like(Y)
oof_w = np.zeros_like(Y)
fids = np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(W, strata, fold_groups)):
    gtr, gva = fold_groups[tr], fold_groups[va]
    mu = Y[tr].mean(0)
    for k, u in enumerate(va):
        mate = tr[gtr == gva[k]]
        oof_t[u] = Y[mate].mean(0) if len(mate) >= 2 else mu
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[tr], Y[tr, j])
        oof_w[va, j] = m.predict(W[va])
    fids[va] = fi + 1
pop = np.tile(Y.mean(0), (n_train, 1))
st, sw, sp = (ndcg5(Y, o) for o in (oof_t, oof_w, pop))
print(f"template-pop={st.mean():.5f} popularity={sp.mean():.5f} word={sw.mean():.5f}")
d = st - sw
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
print(f"template vs word: delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] {'ADOPT' if lo > GATE_FLOOR else 'reject'}")

template-pop=0.37973 popularity=0.39713 word=0.62545
template vs word: delta=-0.24572 CI95=[-0.25375,-0.23813] reject
